Re Ranking Technique

In [1]:
#libraries
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser

loading the text file and splitting into docs

In [2]:
loader = TextLoader("langchain_sample.txt")
raw_docs = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 300,
    chunk_overlap = 50
)
docs = splitter.split_documents(raw_docs)
docs

[Document(metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(metadata={'source': 'langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.'),
 Document(metadata={'source': 'langchain_sample.txt'}, page_content='Retrieval-Augmented Generation (RAG) is a powerful technique where external knowledge is retrieved and passed into the prompt to ground LLM responses. LangChain makes it easy to implement RAG using vector databases like FAISS, Chroma, and Pinecone.'),
 Document(metadata={

Query for testing 

In [3]:
query = "How can i use langchain to build an application with memory and tools?"

Retriever Setup (Dense)

Vectorstore FAISS w HuggingFace Embedding Model

In [4]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embedding_model
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k":8}
)
retriever

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000025066F5A120>, search_kwargs={'k': 8})

defining prompt for reranking and initializing LLM

In [5]:
from langchain.chat_models import init_chat_model
llm = init_chat_model("groq:llama-3.1-8b-instant")
llm

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000250668DAA50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000025067FED940>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [17]:
#prompt template
prompt = PromptTemplate.from_template("""
You are a helpful assitant. Your task is to rank the following documents from most to least relevant to the user's question.

User Question: "{question}"

Documents:
{documents}

Instructions:
 - Think about the relevance of each document to the user's question
 - Return a list of document indices in ranked order, starting from the most relevant

IMPORTANT - RETURN ONLY THE LIST OF INDICES IN SPECIFIED FORMAT, DO NOT RETURN ANY EXPLANATION
                                                           
Output format: comma-separated document indices (e.q., 2,1,3,0,....)
""")

testing out wrt query

In [18]:
retrieved_docs = retriever.invoke(query)
retrieved_docs

[Document(id='70930546-3a09-4433-bf3f-5f490615aaab', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(id='869a643d-7451-413b-a2b2-de9ce61c3470', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.'),
 Document(id='af0176f4-e68a-4720-8a43-efcd6566c884', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond mo

In [19]:
chain = prompt | llm | StrOutputParser()
chain

PromptTemplate(input_variables=['documents', 'question'], input_types={}, partial_variables={}, template='\nYou are a helpful assitant. Your task is to rank the following documents from most to least relevant to the user\'s question.\n\nUser Question: "{question}"\n\nDocuments:\n{documents}\n\nInstructions:\n - Think about the relevance of each document to the user\'s question\n - Return a list of document indices in ranked order, starting from the most relevant\n\nIMPORTANT - RETURN ONLY THE LIST OF INDICES IN SPECIFIED FORMAT, DO NOT RETURN ANY EXPLANATION\n\nOutput format: comma-separated document indices (e.q., 2,1,3,0,....)\n')
| ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000250

In [20]:
doc_lines = [f"{i+1}. {doc.page_content}" for i,doc in enumerate(retrieved_docs)]
formatted_docs = "\n".join(doc_lines)

In [21]:
doc_lines

['1. LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.',
 '2. LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.',
 '3. LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.',
 '4. Memory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.',
 '5. Agents in LangChain are chains that use LLMs to decide which tools to use and in what order. This makes them suitable for multi-s

In [22]:
formatted_docs

'1. LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.\n2. LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.\n3. LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\n4. Memory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.\n5. Agents in LangChain are chains that use LLMs to decide which tools to use and in what order. This makes them suitable for multi-step tasks lik

In [23]:
response = chain.invoke({"question":query,"documents":formatted_docs})
response

'0,1,4,3,5,2,6,7'

PARSING RESPONSE AND RERANKING

In [25]:
indices = [int(x.strip()) for x in response.split(",") if x.strip().isdigit()]
indices

[0, 1, 4, 3, 5, 2, 6, 7]

In [26]:
reranked_docs = [retrieved_docs[i] for i in indices]

FINAL RERANKED DOCS

In [ ]:
print("BEFORE RE-RANKING")
for i,docx in enumerate(retrieved_docs):
    print(f"Rank {i+1}:\n{docx.page_content}")
print("AFTER RE-RANKING")
for i,doc in enumerate(reranked_docs):
    print(f"Rank {i+1}:\n{doc.page_content}")

BEFORE RE-RANKING
Rank 1:
LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.
Rank 2:
LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.
Rank 3:
LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.
Rank 4:
Memory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.
Rank 5:
Agents in LangChain are chains that use LLMs to decide which tools to use and in what order. This makes 